# numbaによる実行時コンパイル

## インストール
Anaconda 2023.09では
numba
(https://numba.pydata.org/)
のupdateが必要かもしれません。
```
pip install -U numba
```

## 用途

Pythonでプログラムを作る際に
Pythonのloopは低速なのがproduction runで問題となるでしょう。
Pythonのloopを高速化する手法にnumbaによる実行時コンパイルを行う手法があります。



## 例

行列積を例とします。

In [ ]:
import numpy as np
import time
from numba import jit

# (N,N)行列

N=400
A = np.random.rand(N,N)
B = np.random.rand(N,N)

In [ ]:
result = [] # 結果を貯める

In [ ]:
def AxB(A,B,N):
    C1 = np.zeros(N*N).reshape(N,N)
    for i in range(N):
        for k in range(N):
            c = 0
            for j in range(N):
                c += A[i,j]*B[j,k]
            C1[i,k] = c
    return C1

t1 = time.time()
C1 = AxB(A,B,N)
t2 = time.time()
result.append(["AxB", t2-t1, 1])
print(t2-t1,"sec")

In [ ]:
@jit(nopython=True)
def AxBjit(A,B,N):
    C = np.zeros(N*N).reshape(N,N)
    for i in range(N):
        for k in range(N):
            c = 0
            for j in range(N):
                c += A[i,j]*B[j,k]
            C[i,k] = c
    return C


In [ ]:
for i in range(5):
    t1 = time.time()
    C2 = AxBjit(A,B,N)
    t2 = time.time()
    print(t2-t1,"sec.", "same result?", np.all(C2==C1))
    result.append(["AxB_jit", t2-t1, i+1])

In [ ]:
# blas使用
t1 = time.time()
C3 = np.dot(A,B)
t2 = time.time()
result.append(["blas3", t2-t1, 1]) 
print(t2-t1,"sec")

In [ ]:
# 結果の表示
import pandas as pd
df = pd.DataFrame(result, columns=["method","elapsed_time","counter"])
df["speedup"] = df.loc[0,"elapsed_time"]/df["elapsed_time"].values
df

一回目はコンパイルを行うので時間がかかりますが、
私のPCではnumbaを用いることで250倍程度の速度向上しました。
(blas level3を用いると更にその8000倍高速に動作します。)

## if文が入った行列積

内部にif文があるとblas level3は使えないので
簡単なif文を入れて速度比を測ります。

In [ ]:
result_if = []

In [ ]:
def AxB_if(A,B,N):
    C1 = np.zeros(N*N).reshape(N,N)
    for i in range(N):
        for k in range(N):
            c = 0
            for j in range(N):
                if A[i,j]>B[j,k]:
                    c += A[i,j]*B[j,k]
            C1[i,k] = c
    return C1

t1 = time.time()
C1_if = AxB(A,B,N)
t2 = time.time()
result_if.append(["AxB_if", t2-t1, 1])
print(t2-t1,"sec")

In [ ]:
@jit(nopython=True)
def AxBif_jit(A,B,N):
    C = np.zeros(N*N).reshape(N,N)
    for i in range(N):
        for k in range(N):
            c = 0
            for j in range(N):
                if A[i,j]>B[j,k]:
                    c += A[i,j]*B[j,k]
            C[i,k] = c
    return C


In [ ]:
for i in range(5):
    t1 = time.time()
    C2_if = AxBjit(A,B,N)
    t2 = time.time()
    print(t2-t1,"sec.", "same result?", np.all(C2_if==C1_if))
    result_if.append(["AxB_if_jit", t2-t1, i+1])

In [ ]:
# 結果の表示
df_if = pd.DataFrame(result_if, columns=["method","elapsed_time","counter"])
df_if["speedup"] = df_if.loc[0,"elapsed_time"]/df_if["elapsed_time"].values
df_if

私のPCでは250倍以上程度の速度向上しました。

# 問題
上のコードで@jit(nopython=True)
を@jitとして実行せよ。NumbaDeprecationWarningが出力されるはずである。

LLMにこの警告メッセージの修正法を尋ねよ。
